# Nero LoRA Training
Trenuje LoRA adapter na wspomnieniach Nero. Wynik: `nero_lora.gguf` do pobrania.


In [ ]:
# 1. Instalacja
!pip install unsloth
!pip install -q llama-cpp-python


In [ ]:
# 2. Wgraj dataset.jsonl z dysku lokalnego
from google.colab import files
uploaded = files.upload()  # wybierz dataset.jsonl


In [ ]:
# 3. Załaduj model z unsloth (4-bit, oszczędza VRAM)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-3-27b-it-bnb-4bit',
    max_seq_length=1024,
    load_in_4bit=True,
)
print('Model zaladowany!')


In [ ]:
# 4. Dodaj LoRA adaptery
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA gotowa!')


In [ ]:
# 5. Przygotuj dataset
import json
from datasets import Dataset

def format_prompt(row):
    return f"""<start_of_turn>system\n{row['system']}<end_of_turn>\n<start_of_turn>user\n{row['instruction']}<end_of_turn>\n<start_of_turn>model\n{row['output']}<end_of_turn>"""

records = []
with open('dataset.jsonl') as f:
    for line in f:
        records.append(json.loads(line))

dataset = Dataset.from_list([{'text': format_prompt(r)} for r in records])
print(f'Dataset: {len(dataset)} probek')


In [ ]:
# 6. Trening
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir='nero_lora_output',
        save_strategy='epoch',
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        report_to='none',
    ),
)
trainer.train()
print('Trening zakonczony!')


In [ ]:
# 7. Eksport do GGUF (kompatybilny z llama-server)
model.save_pretrained_gguf(
    'nero_lora_gguf',
    tokenizer,
    quantization_method='q5_k_m'
)
print('Adapter GGUF gotowy!')
# Pobierz plik
from google.colab import files
import os
for f in os.listdir('nero_lora_gguf'):
    if f.endswith('.gguf'):
        files.download(f'nero_lora_gguf/{f}')
